# SME Finance Copilot — CashPilot-2B QLoRA Fine-Tuning Pipeline

Trains **CashPilot-2B** (`openbmb/MiniCPM5-2B`) on the institutional SME fractional CFO dataset.
- Hardware: Free Kaggle NVIDIA Tesla T4 GPU (16 GB VRAM).
- Setting: In right-hand panel, set **Accelerator** -> **GPU T4** and **Internet** -> **ON**.
- Output: `sme_finance_lora.zip` downloadable from the Output pane for local deployment.

In [ ]:
# 1. Install Fine-Tuning Dependencies
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q peft bitsandbytes trl datasets accelerate

import sys
sys.modules.pop("torchaudio", None)

try:
    import transformers.utils.import_utils as t_utils
    t_utils.is_torchaudio_available = lambda: False
    t_utils._torchaudio_available = False
except Exception:
    pass
print("Dependencies installed successfully.")

In [ ]:
# 2. Verify GPU Accelerator & Unpack Dataset
import gc, os, glob, zipfile, sys, torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.modules.pop("torchaudio", None)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available! Your Kaggle notebook is running on CPU.\n"
        "FIX: In the right sidebar under 'Notebook settings' -> 'Accelerator', select 'GPU T4' (or 'GPU T4 x2') and restart the session."
    )

for obj in ["model", "trainer", "train_dataset", "val_dataset", "tokenizer", "sft_args", "args"]:
    if obj in globals():
        del globals()[obj]
gc.collect()
torch.cuda.empty_cache()

device_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
cc = torch.cuda.get_device_capability()
print(f"GPU Detected: {device_name} ({vram_gb} GB VRAM, Compute Capability: {cc})")

# Extract any dataset zip found in /kaggle/input or /kaggle/working
zip_candidates = (
    glob.glob("/kaggle/input/**/*.zip", recursive=True)
    + glob.glob("/kaggle/working/**/*.zip", recursive=True)
    + glob.glob("./*.zip")
)
for zf in zip_candidates:
    if "lora" in os.path.basename(zf).lower():
        continue
    try:
        with zipfile.ZipFile(zf, "r") as z:
            names = z.namelist()
            if any("finetune_train" in n for n in names):
                print(f"Extracting archive: {zf} -> /kaggle/working/dataset")
                z.extractall("/kaggle/working/dataset")
                break
    except Exception as e:
        print(f"Skipping zip {zf}: {e}")

# Search for train, validation, and tool JSONL files
train_matches = (
    glob.glob("/kaggle/working/**/finetune_train*.jsonl", recursive=True)
    + glob.glob("/kaggle/input/**/finetune_train*.jsonl", recursive=True)
    + glob.glob("./**/finetune_train*.jsonl", recursive=True)
)
val_matches = (
    glob.glob("/kaggle/working/**/finetune_val*.jsonl", recursive=True)
    + glob.glob("/kaggle/input/**/finetune_val*.jsonl", recursive=True)
    + glob.glob("./**/finetune_val*.jsonl", recursive=True)
)
tools_matches = (
    glob.glob("/kaggle/working/**/tools_train*.jsonl", recursive=True)
    + glob.glob("/kaggle/input/**/tools_train*.jsonl", recursive=True)
    + glob.glob("./**/tools_train*.jsonl", recursive=True)
)

if not train_matches:
    all_jsonl = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
    if all_jsonl:
        all_jsonl.sort(key=lambda p: os.path.getsize(p), reverse=True)
        train_path = all_jsonl[0]
        val_path = all_jsonl[1] if len(all_jsonl) > 1 else None
        print(f"Auto-selected largest JSONL as train: {train_path}")
    else:
        raise FileNotFoundError("Could not locate finetune_train.jsonl! Upload sme-finance-dataset.zip to Kaggle.")
else:
    train_path = train_matches[0]
    val_path = val_matches[0] if val_matches else None

tool_path = tools_matches[0] if tools_matches else None

print(f"Train Dataset: {train_path} ({round(os.path.getsize(train_path)/1e6, 2)} MB)")
if val_path:
    print(f"Val Dataset:   {val_path} ({round(os.path.getsize(val_path)/1e6, 2)} MB)")
if tool_path:
    print(f"Tools Dataset: {tool_path} ({round(os.path.getsize(tool_path)/1e6, 2)} MB)")

In [ ]:
# 3. Load 4-Bit NF4 Quantized Model & LoRA Adapters
import os, sys, torch
sys.modules.pop("torchaudio", None)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = os.environ.get("BASE_MODEL_ID", "openbmb/MiniCPM5-2B")
OUTPUT_DIR = "/kaggle/working/sme-finance-lora"
ZIP_OUTPUT = "/kaggle/working/sme_finance_lora.zip"

print(f"Target Base Model: {MODEL_ID}")
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)

# Cast all trainable adapter weights to float32 to ensure numerical stability without GradScaler
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

In [ ]:
# 4. Ingest & Format ChatML Conversations
import json
from datasets import Dataset

train_files = [train_path]
if tool_path:
    train_files.append(tool_path)

def build_chatml_dataset(file_paths, tok):
    texts = []
    skipped = 0
    for fp in file_paths:
        if not fp or not os.path.exists(fp):
            continue
        print(f"Ingesting: {fp}...")
        with open(fp, "r", encoding="utf-8", errors="replace") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                    formatted = tok.apply_chat_template(
                        row["messages"],
                        tools=row.get("tools"),
                        tokenize=False,
                        add_generation_prompt=False,
                    )
                    texts.append(formatted)
                except Exception as ex:
                    skipped += 1
                    if skipped <= 3:
                        print(f"Warning: skipped row {line_no} in {fp}: {ex}")
    print(f"Loaded {len(texts)} formatted dialogues (skipped: {skipped}).")
    return Dataset.from_dict({"text": texts})

print("Formatting datasets into ChatML template...")
train_dataset = build_chatml_dataset(train_files, tokenizer)
val_dataset = build_chatml_dataset([val_path], tokenizer) if val_path else None

In [ ]:
# 5. Execute Supervised Fine-Tuning (~18-25 min on Tesla T4)
from trl import SFTTrainer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

training_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 15,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "fp16": False,                         # Disabled: eliminates AMP GradScaler unscale crash on BFloat16/T4
    "bf16": False,                         # Disabled on T4 (compute capability 7.5 lacks native BF16)
    "optim": "paged_adamw_8bit",
    "report_to": "none",
}

try:
    from trl import SFTConfig
    sft_args = SFTConfig(
        **training_kwargs,
        max_length=1536,
        dataset_text_field="text",
        gradient_checkpointing=True,
    )
    sft_args._n_gpu = 1  # Force single-GPU mode to bypass SFTTrainer DataParallel exception
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_args,
    )
except (ImportError, TypeError):
    from transformers import TrainingArguments
    args = TrainingArguments(
        **training_kwargs,
        gradient_checkpointing=True,
    )
    args._n_gpu = 1  # Force single-GPU mode to bypass SFTTrainer DataParallel exception
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        dataset_text_field="text",
        max_seq_length=1536,
        tokenizer=tokenizer,
        args=args,
    )

print("Starting supervised fine-tuning...")
trainer.train()

In [ ]:
# 6. Save Adapter Weights & Create Download Zip
import os, zipfile
from IPython.display import FileLink, display

print(f"Saving LoRA weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Zipping adapter into {ZIP_OUTPUT}...")
with zipfile.ZipFile(ZIP_OUTPUT, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, OUTPUT_DIR)
            zipf.write(full_path, rel_path)

print(f"SUCCESS! File created: {ZIP_OUTPUT} ({round(os.path.getsize(ZIP_OUTPUT)/1e6, 2)} MB)")
display(FileLink("sme_finance_lora.zip"))

In [ ]:
# 7. Test Native Inference on Trained Adapter
test_prompt = (
    "<|im_start|>system\n"
    "You are the Autonomous SME Finance Copilot fractional CFO.\n"
    "Verified Financial Engine Outputs:\n"
    "- Cash Balance: $65,000.00\n"
    "- Average Daily Burn: $1,250.00/day\n"
    "- Cash Runway: 52 days\n\n"
    "Executive Advisory Guidelines:\n"
    "State the runway and burn directly. Never invent figures.<|im_end|>\n"
    "<|im_start|>user\n"
    "What is our current cash runway?<|im_end|>\n"
    "<|im_start|>assistant\n"
)
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.2, do_sample=False)
print("Model Output:")
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))